# XGBoost ensemble — scheduling dataset

Your reported public scores were **0.99053 scheduling** and **0.99046 hydrology**. Start with the scheduling dataset and test model capacity and probability averaging. No new features are created. The configurations below are experiments, not proven improvements or a promise of 0.995+.

Upload the four matching files from `engineered_scheduling/` to Colab: `hourly_train.csv`, `hourly_test.csv`, `cases.csv`, and `meta.json`. They can be directly in `/content`. Select a GPU runtime. This notebook runs three configurations sequentially on CUDA, keeping predictions in memory for inexpensive blend comparisons.

1. **Select on 2021:** compare your original baseline, a deeper model, a slower shallow model, and four simple probability averages. This needs 42 unit-model fits.
2. **Check on 2022:** refit only the baseline and selected blend's members; compare their micro ROC-AUC. At most 42 more fits. Each training case must finish before validation starts, preserving the seven-day overlap gap.
3. **Optional submission:** refit the selected recipe on all labeled cases. Only the Kaggle CSV is written, with at most 42 additional fits.

Full data and all 14 units are used for the Colab comparisons. No broad random search or repeated local CPU benchmarks are needed. The 2022 period has already influenced earlier experiments, so treat it as a development confirmation, not an untouched final test.

Probability averaging combines models that may make different errors ([scikit-learn ensemble documentation](https://scikit-learn.org/stable/modules/ensemble.html#voting-classifier)). GPU setup follows [XGBoost GPU documentation](https://xgboost.readthedocs.io/en/stable/gpu/).

**CUDA setup:** This notebook uses XGBoost directly with NumPy inputs and a reusable `DMatrix` for prediction; no CuPy installation is needed. `device="cuda"` still selects GPU training and prediction. The small startup test checks the actual selected device and prints driver diagnostics. Removing CuPy does not repair an incompatible NVIDIA driver; the test stops before full training if XGBoost itself cannot use CUDA.

In [ ]:
%pip install -q "xgboost>=2.0"
# CuPy is not required. After changing installed packages, restart the session
# before running the remaining cells.

In [ ]:
from pathlib import Path
import gc
import json
import time
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import roc_auc_score
from threadpoolctl import threadpool_limits
from xgboost import DMatrix, XGBClassifier

DATA_PATH = None  # e.g. Path("/content"); auto-detects the scheduling folder locally.
DEVICE = "cuda"
N_THREADS = 4
HORIZON = 168
SELECTION_YEAR = 2021
CONFIRMATION_YEAR = 2022
RUN_CONFIRMATION = True
RUN_SUBMISSION = False
MIN_GAIN = 0.0001  # Practical selection margin, not statistical significance.
SUBMISSION_PATH = Path("submission_xgboost_ensemble.csv")

BASE_PARAMS = dict(
    objective="binary:logistic", eval_metric="auc", tree_method="hist",
    device=DEVICE, n_estimators=300, learning_rate=0.05, max_depth=5,
    min_child_weight=5, subsample=0.8, colsample_bytree=0.8,
    reg_lambda=1.0, reg_alpha=0.0, random_state=1337, n_jobs=N_THREADS,
)
# Fixed, small experiment: more capacity, or smaller trees with more boosting rounds.
CONFIGS = {
    "baseline": BASE_PARAMS.copy(),
    "deeper": {**BASE_PARAMS, "n_estimators": 600, "learning_rate": 0.035,
               "max_depth": 7, "min_child_weight": 10, "reg_lambda": 5.0},
    "shallow": {**BASE_PARAMS, "n_estimators": 1000, "learning_rate": 0.035,
                "max_depth": 4, "reg_lambda": 2.0},
}
RECIPES = {name: {name: 1.0} for name in CONFIGS}
RECIPES.update({
    "baseline+deeper": {"baseline": 0.5, "deeper": 0.5},
    "baseline+shallow": {"baseline": 0.5, "shallow": 0.5},
    "deeper+shallow": {"deeper": 0.5, "shallow": 0.5},
    "all_three": {name: 1 / 3 for name in CONFIGS},
})
if SELECTION_YEAR >= CONFIRMATION_YEAR:
    raise ValueError("Selection must precede confirmation.")

In [ ]:
# Test XGBoost itself before loading data. NumPy inputs do not require CuPy.
if DEVICE.startswith("cuda"):
    import shutil
    import subprocess
    import xgboost
    print("XGBoost:", xgboost.__version__)
    if shutil.which("nvidia-smi"):
        result = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
        print(result.stdout or result.stderr)
    else:
        print("nvidia-smi unavailable. Check that Colab has a GPU runtime selected.")
    try:
        probe_X = np.array([[0.0], [1.0], [2.0], [3.0]], dtype=np.float32)
        probe = XGBClassifier(device=DEVICE, tree_method="hist", n_estimators=1, max_depth=1)
        probe.fit(probe_X, np.array([0, 0, 1, 1]))
        booster = probe.get_booster()
        actual_device = json.loads(booster.save_config())["learner"]["generic_param"]["device"]
        if not actual_device.startswith("cuda"):
            raise RuntimeError(f"XGBoost selected {actual_device} instead of CUDA.")
        booster.predict(DMatrix(probe_X))
        print(f"XGBoost CUDA training and prediction verified: {actual_device}")
        del probe, booster, probe_X
    except Exception as error:
        raise RuntimeError(
            "XGBoost could not use CUDA. Select a Colab GPU runtime and restart the session "
            "after package changes. If this persists, share the nvidia-smi output and XGBoost "
            "version printed above to check driver compatibility. No full training has started. "
            f"Original error: {error}"
        ) from error
elif DEVICE != "cpu":
    raise ValueError("Set DEVICE to 'cuda' or 'cpu'.")

In [ ]:
if DATA_PATH is None:
    folders = (Path("engineered_scheduling"), Path("../engineered_scheduling"), Path("."))
    DATA_PATH = next((p for p in folders if (p / "hourly_train.csv").is_file()), None)
if DATA_PATH is None:
    raise FileNotFoundError("Upload the scheduling CSVs and meta.json together, or set DATA_PATH.")
DATA_PATH = Path(DATA_PATH).resolve()
print(f"Dataset: {DATA_PATH}")
meta = json.loads((DATA_PATH / "meta.json").read_text())
UNITS, TARGETS = meta["units"], meta["y_cols"]
FEATS = [c for c in meta["hourly_feature_cols"] + meta["static_feature_cols"]
         if not c.startswith("dvol_")]
if meta.get("feature_engineering", {}).get("variant") != "scheduling_context":
    raise ValueError("Use the matching engineered_scheduling CSVs and meta.json for this comparison.")
if (TARGETS != [f"y_{u}" for u in UNITS] or len(FEATS) != len(set(FEATS))
        or set(FEATS).intersection(TARGETS + ["Run No", "starttime", "split", "fold"])):
    raise ValueError("Feature/target metadata is inconsistent.")

cases = pd.read_csv((DATA_PATH / "cases.csv"), dtype={c: np.float32 for c in meta["static_feature_cols"]})
cases["starttime"] = pd.to_datetime(cases["starttime"], utc=True)
if cases["Run No"].isna().any() or cases["Run No"].duplicated().any() or cases["starttime"].isna().any():
    raise ValueError("cases.csv must have unique Run No values and valid start times.")


def load_hourly(path, split):
    targets = TARGETS if split == "train" else []
    dtypes = {c: np.float32 for c in meta["hourly_feature_cols"] + targets}
    dtypes.update({"Run No": np.int64, "t": np.int16})
    hourly = pd.read_csv(path, dtype=dtypes)
    expected_runs = set(cases.loc[cases["split"].eq(split), "Run No"])
    if set(hourly["Run No"]) != expected_runs or hourly.duplicated(["Run No", "t"]).any():
        raise ValueError(f"{path.name}: missing/unexpected cases or duplicate (Run No, t) rows.")
    rows = hourly.merge(cases, on="Run No", how="left", validate="many_to_one")
    rows = rows.sort_values(["Run No", "t"]).reset_index(drop=True)
    if (not rows.groupby("Run No").size().eq(HORIZON).all()
            or not np.array_equal(rows["t"], np.tile(np.arange(HORIZON), len(expected_runs)))):
        raise ValueError(f"{path.name}: each case must have exactly the 168 hours t=0..167.")
    if not np.isfinite(rows[FEATS].to_numpy(dtype=np.float32)).all():
        raise ValueError(f"{path.name}: missing or infinite feature values.")
    return rows


train = load_hourly((DATA_PATH / "hourly_train.csv"), "train")
if not np.isin(train[TARGETS].to_numpy(), [0, 1]).all():
    raise ValueError("Training labels must be 0 or 1.")
row_info = train[["Run No", "t", "starttime"]].copy()
X = train[FEATS].to_numpy(dtype=np.float32)
y = train[TARGETS].to_numpy(dtype=np.uint8)
del train  # Keep the arrays and identifiers; avoid retaining a second feature matrix.
print(f"Loaded {row_info['Run No'].nunique():,} cases | {len(X):,} hourly rows | {len(FEATS)} features | {len(UNITS)} units")

In [ ]:
def micro_auc(labels, probabilities):
    return roc_auc_score(labels.ravel(), probabilities.ravel())


def year_masks(year):
    start = pd.Timestamp(f"{year}-01-01", tz="UTC")
    end = pd.Timestamp(f"{year + 1}-01-01", tz="UTC")
    dates = row_info["starttime"]
    fit = (dates + pd.Timedelta(hours=HORIZON) <= start).to_numpy()
    valid = ((dates >= start) & (dates < end)).to_numpy()
    if not fit.any() or not valid.any():
        raise ValueError(f"Missing training or validation cases for {year}.")
    print(f"{year}: {fit.sum() // HORIZON:,} training cases; {valid.sum() // HORIZON:,} validation cases")
    return fit, valid


def fit_members(X_fit, y_fit, X_predict, members):
    # XGBoost manages GPU transfers internally; CuPy is unnecessary.
    # Reuse the prediction matrix and fit models sequentially on one GPU.
    predict_matrix = DMatrix(X_predict, nthread=N_THREADS)
    predictions = {}
    with threadpool_limits(limits=N_THREADS):
        for name in members:
            started = time.perf_counter()
            probabilities = np.empty((len(X_predict), len(UNITS)), dtype=np.float32)
            for j, unit in enumerate(UNITS):
                classes = np.unique(y_fit[:, j])
                if len(classes) == 1:
                    probabilities[:, j] = classes[0]
                else:
                    model = XGBClassifier(**CONFIGS[name])
                    model.fit(X_fit, y_fit[:, j])
                    probabilities[:, j] = model.get_booster().predict(predict_matrix)
                    del model
                print(f"  {name}: {j + 1}/{len(UNITS)} {unit}", flush=True)
            predictions[name] = probabilities
            print(f"{name} completed in {time.perf_counter() - started:.1f}s", flush=True)
    del predict_matrix
    gc.collect()
    return predictions


def blend(predictions, weights):
    return sum(weight * predictions[name] for name, weight in weights.items())

## Select a model or blend on 2021

The same fixed configurations train on the same rows. Seven recipes share just three sets of predictions, so testing averages needs no extra training. Selection uses **micro ROC-AUC across all units**, rather than averaging per-unit scores. Keep the baseline if the winning gain is less than `MIN_GAIN`.

Tree counts are fixed in advance. This avoids using the confirmation labels for early stopping. Larger models can still overfit; the later comparison is part of deciding whether to adopt them.

In [ ]:
fit, valid = year_masks(SELECTION_YEAR)
selection_predictions = fit_members(X[fit], y[fit], X[valid], CONFIGS)
selection_results = pd.DataFrame([
    {"recipe": name, "micro_auc": micro_auc(y[valid], blend(selection_predictions, weights))}
    for name, weights in RECIPES.items()
]).set_index("recipe")
selection_results["gain_over_baseline"] = selection_results["micro_auc"] - selection_results.loc["baseline", "micro_auc"]
selection_results = selection_results.sort_values("micro_auc", ascending=False, kind="stable")
selected_recipe = selection_results.index[0]
if selection_results.loc[selected_recipe, "gain_over_baseline"] < MIN_GAIN:
    selected_recipe = "baseline"
selected_weights = RECIPES[selected_recipe].copy()
# Reset downstream state whenever this selection cell is rerun.
recommended_recipe = "baseline"
confirmation_results = None
print("Selected recipe:", selected_recipe, selected_weights)
display(selection_results.round(6))
print("Selected hyperparameters:")
print(json.dumps({name: CONFIGS[name] for name in selected_weights}, indent=2))

## Compare the frozen recipe on 2022

Only selected members and the baseline are fitted here. Blend weights stay fixed from 2021. The table reports their complete 14-unit micro ROC-AUC. The notebook recommends retaining the baseline unless the selected recipe improves by at least `MIN_GAIN` in this comparison too. This is a conservative decision rule, not proof of a leaderboard gain. No per-unit model choices are made from these labels.

In [ ]:
recommended_recipe = "baseline"
confirmation_results = None
if RUN_CONFIRMATION:
    fit, valid = year_masks(CONFIRMATION_YEAR)
    members = list(dict.fromkeys(["baseline", *selected_weights]))
    confirmation_predictions = fit_members(X[fit], y[fit], X[valid], members)
    candidate = blend(confirmation_predictions, selected_weights)
    baseline_auc = micro_auc(y[valid], confirmation_predictions["baseline"])
    candidate_auc = micro_auc(y[valid], candidate)
    confirmation_results = pd.DataFrame([
        {"recipe": "baseline", "micro_auc": baseline_auc},
        {"recipe": selected_recipe, "micro_auc": candidate_auc},
    ]).drop_duplicates("recipe").set_index("recipe")
    confirmation_results["gain_over_baseline"] = confirmation_results["micro_auc"] - baseline_auc
    if candidate_auc - baseline_auc >= MIN_GAIN:
        recommended_recipe = selected_recipe
    display(confirmation_results.round(6))
    unit_scores = pd.DataFrame({
        "baseline": [roc_auc_score(y[valid, j], confirmation_predictions["baseline"][:, j])
                     if np.unique(y[valid, j]).size == 2 else np.nan for j in range(len(UNITS))],
        "selected": [roc_auc_score(y[valid, j], candidate[:, j])
                     if np.unique(y[valid, j]).size == 2 else np.nan for j in range(len(UNITS))],
    }, index=UNITS)
    display(unit_scores.round(5))
    print("Recommended recipe:", recommended_recipe, RECIPES[recommended_recipe])
else:
    print("Confirmation skipped; the default submission recipe remains the baseline.")

## Optional Kaggle CSV

Set `RUN_SUBMISSION = True` in the next cell when ready. By default, it uses `recommended_recipe`. You can set `SUBMISSION_RECIPE` explicitly to any key in `RECIPES`, for example `"baseline+deeper"`, to test that recipe instead. The chosen weights and parameters print before training.

Fresh models use all labeled cases, including prior validation and gap cases, once model selection is finished. Test predictions remain probabilities. The original unit/hour ordering comes from the matching metadata. The inflow source file is unnecessary; only the four scheduling files are needed. No models, search reports, or auxiliary CSVs are saved.

In [ ]:
# RUN_SUBMISSION = True
SUBMISSION_RECIPE = recommended_recipe
if RUN_SUBMISSION:
    weights = RECIPES[SUBMISSION_RECIPE]
    print("Submission recipe:", SUBMISSION_RECIPE, weights)
    print(json.dumps({name: CONFIGS[name] for name in weights}, indent=2))
    test = load_hourly(DATA_PATH / "hourly_test.csv", "test")
    test_run_nos = test["Run No"].drop_duplicates().to_numpy()
    X_test = test[FEATS].to_numpy(dtype=np.float32)
    del test
    test_predictions = fit_members(X, y, X_test, weights)
    probabilities = blend(test_predictions, weights)
    wide = probabilities.reshape(len(test_run_nos), HORIZON, len(UNITS))
    wide = wide.transpose(0, 2, 1).reshape(len(test_run_nos), -1)
    columns = [f"result_committed_{unit}_t{hour}" for unit in UNITS for hour in range(HORIZON)]
    submission = pd.DataFrame(wide, columns=columns)
    submission.insert(0, "Run No", test_run_nos)
    if set(submission.columns) != set(meta["submission_columns"]):
        raise ValueError("Submission columns do not match the dataset metadata.")
    if not np.isfinite(wide).all() or ((wide < 0) | (wide > 1)).any():
        raise ValueError("Submission probabilities must be finite and between zero and one.")
    submission = submission[meta["submission_columns"]]
    submission.to_csv(SUBMISSION_PATH, index=False)
    print(f"Saved {SUBMISSION_PATH}: {submission.shape}")
    display(submission.iloc[:5, :6])
else:
    print("Submission skipped. Set RUN_SUBMISSION = True above when ready.")